# HOWTO: Simple Trees used for Search-based AI in Python


To understand AI search trees, you need to understand how tree data structures in general work. If you are not sure, then you can watch 
the video [Binary Search Trees Python Tutorial](https://www.youtube.com/watch?v=pkYVOmU3MgA)
from the course Data Structures and Algorithms in Python - Full Course for Beginners.

Search trees used in AI slightly differ from the usual binary search trees taught in data structures courses. Here are some key differences:

* AI search trees are dynamically built during search. Branches no longer needed during 
    search may be removed to save memory (e.g., in DFS).
* AI search trees are typically not binary trees, but nodes can have many children
    (often representing actions).
* For AI search trees we typically do not use links from parents to children, but 
    child notes have a link (references in Python) to their parent. The reason is that going down the tree, new nodes 
    are creates so we automatically have a reference. Links are only used to go back up the tree.

Here is an implementation of the basic node structure for an AI search tree (see Fig 3.7 on page 73). The state in the example is a position represented as `(row, col)`. I have added a method that extracts the path from the root node to the current node. It can be used to get the path when the search is completed.

In [1]:
class Node:
    def __init__(self, pos, parent, action, cost):
        self.pos = tuple(pos)    # the state; positions are (row,col)
        self.parent = parent     # reference to parent node. None means root node.
        self.action = action     # action used in the transition function (root node has None)
        self.cost = cost         # for uniform cost this is the depth. It is also g(n) for A* search

    def __str__(self):
        """print the node in a human readable format"""
        return f"Node - pos = {self.pos}; action = {self.action}; cost = {self.cost}"
    
    def get_path_from_root(self):
        """returns nodes on the path from the root to the current node."""
        node = self
        path = [node]
    
        while not node.parent is None:
            node = node.parent
            path.append(node)
        
        path.reverse()
        
        return(path)

## Example

Create the root node and append the node for going east with a cost of 1. The example also illustrates memory management using reference counts and garbage collection (see [Python references](https://www.pythontutorial.net/advanced-python/python-references/)).

In [2]:
# import sys to access reference counts
import sys

Create the root node for position (0,0), no action was needed to get to the state represented by the root node, and the path cost to get there is 0.

In [3]:
root = Node(pos = (0,0), parent = None, action = None, cost = 0)
print("root:", root)
print("root (reference):", repr(root))
# Note: -1 is used because passing root to the getrefcount function adds a reference
print("Root's reference count for garbage collection:", sys.getrefcount(root) - 1) 

root: Node - pos = (0, 0); action = None; cost = 0
root (reference): <__main__.Node object at 0x7f0f682f8440>
Root's reference count for garbage collection: 1


Go east with cost 1 (from the parent root).

In [4]:
node2 = Node(pos = (0,1), parent = root, action = "E", cost = 1)
print("node2:", node2)
print("parent of node2: ", node2.parent)
print("parent of node2 (reference): ", repr(node2.parent))
print("Root is now referenced two times (root variable, parent of node2). Reference count for garbage collection (root node):", sys.getrefcount(root) -1 )

node2: Node - pos = (0, 1); action = E; cost = 1
parent of node2:  Node - pos = (0, 0); action = None; cost = 0
parent of node2 (reference):  <__main__.Node object at 0x7f0f682f8440>
Root is now referenced two times (root variable, parent of node2). Reference count for garbage collection (root node): 2


Note that the root node is safe from garbage collection as long as we have also a reference in node2 (parent).

In [5]:
root = None
print(root)
print("parent of node2 (reference to root node): ", node2.parent)
print("parent of node2 (reference to root node): ", repr(node2.parent))
print("Reference count for garbage collection (root node):", sys.getrefcount(node2.parent)-1)

None
parent of node2 (reference to root node):  Node - pos = (0, 0); action = None; cost = 0
parent of node2 (reference to root node):  <__main__.Node object at 0x7f0f682f8440>
Reference count for garbage collection (root node): 1


Path from root to node2

In [6]:
path = node2.get_path_from_root()
print("References:", path)
print("Positions:", [n.pos for n in path])
print("Actions:", [n.action for n in path])
print("Path Cost:", [n.cost for n in path])

References: [<__main__.Node object at 0x7f0f682f8440>, <__main__.Node object at 0x7f0f682ec910>]
Positions: [(0, 0), (0, 1)]
Actions: [None, 'E']
Path Cost: [0, 1]


Once we delete the reference to node2, the reference count for all nodes goes to zero and the whole tree is exposed to garbage collection.

In [7]:
node2 = None

---

# BÀI 1. CẤU TRÚC CÂY TÌM KIẾM VÀ LỚP `Node`

**Mục tiêu:** Quan sát các thuộc tính của nút tìm kiếm, giải thích đặc điểm tạo động và không nhị phân của cây tìm kiếm AI, và sử dụng `get_path_from_root()` để truy hồi đường đi.

**Lưu ý:** Phần trên của notebook là ví dụ giảng viên cung cấp; phần dưới là bài thực hành bổ sung. Sử dụng lại lớp `Node` đã được khai báo trong notebook gốc.

## 1. Quan sát bốn thuộc tính của lớp `Node`

Lớp `Node` biểu diễn **một nút trong cây tìm kiếm**, với bốn thuộc tính:

| Thuộc tính | Ý nghĩa | Ví dụ |
|---|---|---|
| `pos` | Trạng thái tại nút, thể hiện bằng tọa độ `(row, col)`. | `(2, 3)` |
| `parent` | Tham chiếu tới nút cha; nút gốc có `parent = None`. | Nút ở `(2, 2)` |
| `action` | Hành động đưa từ nút cha đến nút hiện tại; nút gốc không có hành động. | `"E"` (East – sang phải) |
| `cost` | Tổng chi phí đường đi từ gốc đến nút hiện tại, tương ứng `g(n)` trong A*. | `1` |

Trong ví dụ này, mỗi bước di chuyển có chi phí bằng **1**. Ta tạo một cây với **ba nhánh xuất phát từ nút gốc** để quan sát rõ tính chất không nhị phân.

In [8]:
# Sử dụng lớp Node đã định nghĩa ở các cell đầu của notebook
# Trạng thái là (row, col); N: lên, E: phải, S: xuống
root_demo = Node(pos=(2, 2), parent=None, action=None, cost=0)
node_n = Node(pos=(1, 2), parent=root_demo, action="N", cost=1)
node_e = Node(pos=(2, 3), parent=root_demo, action="E", cost=1)
node_s = Node(pos=(3, 2), parent=root_demo, action="S", cost=1)

# Mở rộng thêm nhánh E -> E -> S
node_ee = Node(pos=(2, 4), parent=node_e, action="E", cost=2)
node_ees = Node(pos=(3, 4), parent=node_ee, action="S", cost=3)

nodes = [
    ("root_demo", root_demo),
    ("node_n", node_n),
    ("node_e", node_e),
    ("node_s", node_s),
    ("node_ee", node_ee),
    ("node_ees", node_ees),
]

print(f"{'Node':<12} {'pos':<10} {'parent.pos':<12} {'action':<10} {'cost':<5}")
print("-" * 55)
for name, node in nodes:
    parent_pos = node.parent.pos if node.parent is not None else None
    print(f"{name:<12} {str(node.pos):<10} {str(parent_pos):<12} "
          f"{str(node.action):<10} {node.cost:<5}")

Node         pos        parent.pos   action     cost 
-------------------------------------------------------
root_demo    (2, 2)     None         None       0    
node_n       (1, 2)     (2, 2)       N          1    
node_e       (2, 3)     (2, 2)       E          1    
node_s       (3, 2)     (2, 2)       S          1    
node_ee      (2, 4)     (2, 3)       E          2    
node_ees     (3, 4)     (2, 4)       S          3    


### Nhận xét về các thuộc tính

- **Nút gốc** `root_demo` có `pos = (2, 2)`, `parent = None`, `action = None`, `cost = 0` vì chưa có hành động hoặc chi phí di chuyển.
- **Nút con** có `parent` tham chiếu đến nút sinh ra nó. Ví dụ `node_e.parent` là `root_demo`, còn `node_ee.parent` là `node_e`.
- **`action`** ghi lại hành động dùng để di chuyển từ nút cha đến nút hiện tại. Ví dụ `node_ees.action = "S"`.
- **`cost`** tăng dần theo tổng chi phí trên đường đi. Với mỗi bước có chi phí bằng 1, `node_ees.cost = 3`.

### 2. Vì sao cây tìm kiếm AI được tạo động và không nhất thiết là cây nhị phân?

**Cây tìm kiếm được tạo động:** Các nút được sinh ra dần trong quá trình thuật toán mở rộng các trạng thái đang xét, thay vì phải tạo trước toàn bộ cây. Mỗi khi mở rộng một nút, thuật toán xét các hành động hợp lệ để tạo những nút con tương ứng. Cách này giúp tập trung tài nguyên vào các nhánh cần khảo sát; các nhánh không còn được tham chiếu có thể được Python thu hồi bộ nhớ.

**Cây tìm kiếm không nhất thiết là cây nhị phân:** Số nút con phụ thuộc vào số **hành động hợp lệ** tại từng trạng thái, chứ không bị giới hạn ở hai nút con. Trong ví dụ, nút `(2, 2)` có ba nút con ứng với các hành động `N`, `E`, `S`. Ở một mê cung di chuyển bốn hướng, một trạng thái có thể có tới bốn lựa chọn trước khi xét tường và biên bản đồ.

```text
                   (2, 2)  [root]
                  /   |   \
               N /  E |    \ S
                /     |     \
            (1, 2) (2, 3) (3, 2)
                      |
                    E |
                      v
                    (2, 4)
                      |
                    S |
                      v
                    (3, 4)
```

*Khác với cách biểu diễn cây truyền thống lưu danh sách con tại mỗi nút, lớp `Node` trong notebook lưu liên kết từ nút con lên `parent`. Nhờ vậy ta truy hồi đường đi từ nút kết quả về nút gốc.*

## 3. Truy hồi dãy trạng thái và hành động bằng `get_path_from_root()`

Phương thức có sẵn `get_path_from_root()` bắt đầu từ nút hiện tại, liên tục đi theo thuộc tính `parent` đến nút gốc, sau đó đảo ngược danh sách để **trả kết quả theo thứ tự từ gốc đến nút hiện tại**.

Trong ví dụ này, ta truy hồi từ `node_ees` về gốc (bên trong hàm) và hiển thị đường đi theo chiều từ gốc đến `node_ees`. Hành động `None` ở nút gốc không được tính là bước di chuyển.

In [9]:
# Truy hồi các nút trên đường đi từ nút gốc tới node_ees
path_demo = node_ees.get_path_from_root()

positions = [node.pos for node in path_demo]
actions = [node.action for node in path_demo if node.action is not None]
accumulated_costs = [node.cost for node in path_demo]

print("Dãy trạng thái:", positions)
print("Dãy hành động:", actions)
print("Chi phí tích lũy tại mỗi nút:", accumulated_costs)
print("Tổng chi phí đường đi:", path_demo[-1].cost)
print("Số bước di chuyển:", len(actions))

# Đối chiếu kết quả mong đợi
assert positions == [(2, 2), (2, 3), (2, 4), (3, 4)]
assert actions == ["E", "E", "S"]
assert accumulated_costs == [0, 1, 2, 3]
assert path_demo[-1].cost == 3
print("Kiểm tra kết quả: ĐẠT")

Dãy trạng thái: [(2, 2), (2, 3), (2, 4), (3, 4)]
Dãy hành động: ['E', 'E', 'S']
Chi phí tích lũy tại mỗi nút: [0, 1, 2, 3]
Tổng chi phí đường đi: 3
Số bước di chuyển: 3
Kiểm tra kết quả: ĐẠT


### Kết quả và kết luận

- **Dãy trạng thái (gốc → đích):** `(2, 2) → (2, 3) → (2, 4) → (3, 4)`.
- **Dãy hành động:** `E → E → S` (phải → phải → xuống).
- **Chi phí tích lũy:** `0 → 1 → 2 → 3`; **tổng chi phí = 3** trong giả định mỗi bước tốn 1 đơn vị.
- **Số bước di chuyển:** `3`.

**Kết luận:** Lớp `Node` lưu trạng thái, nút cha, hành động và chi phí tích lũy. Cây tìm kiếm được mở rộng khi cần và số nút con thay đổi theo các hành động hợp lệ. Liên kết `parent` cho phép hàm `get_path_from_root()` khôi phục đường đi và chuỗi hành động, hỗ trợ các thuật toán tìm kiếm như BFS, DFS và A*.

> **Ghi chú về cell minh họa bộ nhớ trong notebook gốc:** Sau khi đặt `node2 = None`, các nút chưa chắc đã được thu hồi ngay nếu vẫn còn tham chiếu khác (chẳng hạn biến `path` lưu danh sách nút). Việc Python thu hồi đối tượng phụ thuộc vào các tham chiếu còn tồn tại.